# Phase 2: Coffee Shop Sales Data Cleaning & Preprocessing

This notebook demonstrates data cleaning, missing value handling, outlier detection, and datetime preprocessing on the synthetic coffee shop sales dataset.

### Step 1: Import Required Libraries and Load Data
Import `pandas` and `numpy`, then read `coffee_shop_sales.csv`.

In [ ]:
# Import pandas for data processing
import pandas as pd

# Import numpy for numerical computations
import numpy as np

# Load raw dataset generated in Phase 1
df = pd.read_csv('coffee_shop_sales.csv')

# Display initial shape and first few rows
print("Initial Dataset Shape:", df.shape)
df.head()

### Step 2: Inject Synthetic Noise (Missing Values & Outliers)
To simulate real-world data quality issues, we intentionally introduce NaNs and extreme outlier values into the dataset.

In [ ]:
# Set seed for reproducible noise generation
np.random.seed(101)
n_rows = len(df)

# Inject random missing values (NaNs) into customer_age, satisfaction_score, and payment_method
nan_age_idx = np.random.choice(n_rows, size=50, replace=False)
nan_score_idx = np.random.choice(n_rows, size=40, replace=False)
nan_pay_idx = np.random.choice(n_rows, size=30, replace=False)

df.loc[nan_age_idx, 'customer_age'] = np.nan
df.loc[nan_score_idx, 'satisfaction_score'] = np.nan
df.loc[nan_pay_idx, 'payment_method'] = np.nan

# Inject extreme outliers into customer_age and price
outlier_age_idx = np.random.choice(n_rows, size=5, replace=False)
outlier_price_idx = np.random.choice(n_rows, size=5, replace=False)

df.loc[outlier_age_idx, 'customer_age'] = [150, 140, -5, 200, -10]
df.loc[outlier_price_idx, 'price'] = [499.99, 750.00, 999.00, 550.00, 899.99]
df.loc[outlier_price_idx, 'total_amount'] = df.loc[outlier_price_idx, 'price'] * df.loc[outlier_price_idx, 'quantity']

print("Synthetic noise injected successfully.")

### Step 3: Data Inspection & Missing Value Detection
Examine data types, non-null counts, and count missing values per column.

In [ ]:
# Check column data types and missing counts
print("=== DataFrame Summary ===")
df.info()

# Calculate missing values per column
missing_summary = df.isnull().sum()
print("\n=== Missing Values Per Column ===")
print(missing_summary[missing_summary > 0])

### Step 4: Handle Missing Values (Imputation)
Impute missing numerical values using median and categorical values using mode.

In [ ]:
# Calculate valid median age (excluding extreme outliers)
valid_age_median = df['customer_age'][(df['customer_age'] >= 18) & (df['customer_age'] <= 100)].median()

# Impute missing customer_age values with median age
df['customer_age'] = df['customer_age'].fillna(valid_age_median)

# Impute missing satisfaction_score values with median score
median_satisfaction = df['satisfaction_score'].median()
df['satisfaction_score'] = df['satisfaction_score'].fillna(median_satisfaction)

# Impute missing payment_method values with the most frequent value (mode)
most_frequent_payment = df['payment_method'].mode()[0]
df['payment_method'] = df['payment_method'].fillna(most_frequent_payment)

# Verify missing values after imputation
print("Remaining missing values:", df.isnull().sum().sum())

### Step 5: Outlier Detection and Handling
Detect impossible customer ages and extreme price outliers using IQR criteria, then cap/fix them.

In [ ]:
# Fix impossible customer ages (< 18 or > 100) by assigning median age
invalid_age_mask = (df['customer_age'] < 18) | (df['customer_age'] > 100)
df.loc[invalid_age_mask, 'customer_age'] = valid_age_median

# Calculate IQR for price column to identify extreme price outliers
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
upper_threshold = Q3 + 3 * IQR

# Replace extreme prices above threshold with Q3 price value
extreme_price_mask = df['price'] > upper_threshold
df.loc[extreme_price_mask, 'price'] = Q3

# Recalculate total_amount based on cleaned price
df['total_amount'] = np.round(df['price'] * df['quantity'], 2)

# Summary statistics after outlier treatment
display(df[['price', 'customer_age', 'total_amount']].describe())

### Step 6: Datetime Conversion & Feature Engineering
Convert `transaction_date` to `datetime` objects and extract date/time components.

In [ ]:
# Convert transaction_date string column to pandas Datetime format
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract datetime features
df['transaction_year'] = df['transaction_date'].dt.year
df['transaction_month'] = df['transaction_date'].dt.month
df['transaction_day'] = df['transaction_date'].dt.day
df['transaction_hour'] = df['transaction_date'].dt.hour
df['day_of_week'] = df['transaction_date'].dt.day_name()

# Preview updated DataFrame with date features
df[['transaction_date', 'transaction_month', 'transaction_hour', 'day_of_week']].head()

### Step 7: Export Cleaned Dataset
Validate cleaned dataset shape, data types, and export to `coffee_shop_sales_cleaned.csv`.

In [ ]:
# Check final info and null count
print("Final Cleaned DataFrame Info:")
df.info()

# Export cleaned dataset to CSV
output_cleaned_path = 'coffee_shop_sales_cleaned.csv'
df.to_csv(output_cleaned_path, index=False)
print(f"Successfully saved clean dataset to '{output_cleaned_path}' with shape {df.shape}")